In [ ]:
"""
=============================================================
  CNN-Based Facial Recognition System  (v2 – DeconvSkip Head)
  Uses: FaceNet (InceptionResnetV1) + MTCNN + OpenCV
  Modification: Adds a small deconv+skip refinement block on
                top of the frozen FaceNet features before the
                SVM classifier.  Trains fast, same pipeline.
  Saves model as: model3.pkl
=============================================================
"""

import os
import torch
import torch.nn as nn
import cv2
import numpy as np
import pickle
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────
FACES_DIR = "known_faces_folder"               # Root folder containing subfolders per person
MODEL_SAVE_PATH = "model3.pkl"                 # Path for saved classifier
CONFIDENCE_THRESHOLD = 0.55                    # Below this → "Unknown"
IMAGE_SIZE = (160, 160)                        # FaceNet input size
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────
#  DECONV-SKIP REFINEMENT HEAD
# ─────────────────────────────────────────────
class DeconvSkipHead(nn.Module):
    """
    Lightweight refinement head with deconv layers and skip connections.
    """

    def __init__(self, embed_dim=512):
        super().__init__()
        self.embed_dim = embed_dim

        # Project 512 → 512 (32 channels * 4 * 4)
        self.fc_in = nn.Linear(embed_dim, 32 * 4 * 4)
        self.bn_in = nn.BatchNorm1d(32 * 4 * 4)

        # Deconv block 1: 32ch 4×4 → 64ch 8×8
        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Deconv block 2: 64ch 8×8 → 32ch 16×16
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # Skip: upsample 32ch 4×4 → 32ch 16×16 and merge
        self.skip_up = nn.Upsample(size=(16, 16), mode="bilinear", align_corners=False)

        # After concat skip (32 + 32 = 64 channels) → merge to 32
        self.merge = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # Final pooling + projection back to 512
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_out = nn.Linear(32, embed_dim)

        # Residual gate (learnable scaling of the refinement)
        self.gate = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        """x: (B, 512) raw FaceNet embedding."""
        identity = x  # original skip

        h = self.fc_in(x)
        h = self.bn_in(h)
        h = torch.relu(h)
        h = h.view(-1, 32, 4, 4)  # (B, 32, 4, 4)

        spatial_skip = h  

        h = self.deconv1(h)   # (B, 64, 8, 8)
        h = self.deconv2(h)   # (B, 32, 16, 16)

        # Spatial skip connection
        skip_up = self.skip_up(spatial_skip) 
        h = torch.cat([h, skip_up], dim=1)  # (B, 64, 16, 16)
        h = self.merge(h)  # (B, 32, 16, 16)

        h = self.pool(h).flatten(1)  # (B, 32)
        h = self.fc_out(h)           # (B, 512)

        # Gated residual
        out = identity + self.gate * h
        return out

# ─────────────────────────────────────────────
#  INITIALIZE MODELS
# ─────────────────────────────────────────────
def load_models():
    mtcnn = MTCNN(
        image_size=160,
        margin=20,
        min_face_size=40,
        thresholds=[0.6, 0.7, 0.7],
        factor=0.709,
        post_process=True,
        keep_all=False,
        device=DEVICE
    )
    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
    head = DeconvSkipHead(embed_dim=512).eval().to(DEVICE)
    print("[INFO] DeconvSkipHead loaded.")
    return mtcnn, resnet, head

# ─────────────────────────────────────────────
#  EMBEDDING EXTRACTION
# ─────────────────────────────────────────────
def get_embedding(mtcnn, resnet, head, img_rgb: np.ndarray):
    pil_img = Image.fromarray(img_rgb)
    face_tensor = mtcnn(pil_img)

    if face_tensor is None:
        return None

    face_tensor = face_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        raw_emb = resnet(face_tensor)    
        refined_emb = head(raw_emb)      

    return refined_emb.squeeze().cpu().numpy()

# ─────────────────────────────────────────────
#  TRAINING PHASE
# ─────────────────────────────────────────────
def train(faces_dir: str, mtcnn, resnet, head):
    embeddings = []
    labels = []
    failed = 0

    if not os.path.exists(faces_dir):
        raise FileNotFoundError(f"Directory '{faces_dir}' not found.")

    persons = [p for p in os.listdir(faces_dir) if os.path.isdir(os.path.join(faces_dir, p))]

    if not persons:
        raise ValueError(f"No subfolders found in '{faces_dir}'.")

    print(f"\n[TRAIN] Found {len(persons)} person(s): {persons}")

    for person in persons:
        person_dir = os.path.join(faces_dir, person)
        img_files = [f for f in os.listdir(person_dir) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]

        print(f"  → {person}: {len(img_files)} image(s)")

        for img_file in img_files:
            img_path = os.path.join(person_dir, img_file)
            img_bgr = cv2.imread(img_path)

            if img_bgr is None:
                failed += 1
                continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            emb = get_embedding(mtcnn, resnet, head, img_rgb)

            if emb is None:
                failed += 1
                continue

            embeddings.append(emb)
            labels.append(person)

    if len(embeddings) == 0:
        raise RuntimeError("No valid face embeddings extracted.")

    le = LabelEncoder()
    y = le.fit_transform(labels)
    X = np.array(embeddings)

    if len(set(y)) > 1 and len(X) > 4:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    else:
        X_train, X_test, y_train, y_test = X, X, y, y

    clf = SVC(kernel="linear", probability=True, C=1.0)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print("\n[TRAIN] Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    model_data = {"clf": clf, "le": le}
    with open(MODEL_SAVE_PATH, "wb") as f:
        pickle.dump(model_data, f)

    print(f"[TRAIN] Model saved → {MODEL_SAVE_PATH}")
    return clf, le

# ─────────────────────────────────────────────
#  PREDICTION & RECOGNITION UTILS
# ─────────────────────────────────────────────
def load_classifier():
    if not os.path.exists(MODEL_SAVE_PATH):
        raise FileNotFoundError(f"Model '{MODEL_SAVE_PATH}' not found.")
    with open(MODEL_SAVE_PATH, "rb") as f:
        data = pickle.load(f)
    return data["clf"], data["le"]

def predict_face(emb: np.ndarray, clf: SVC, le: LabelEncoder):
    probs = clf.predict_proba([emb])[0]
    max_prob = np.max(probs)
    pred_idx = np.argmax(probs)
    name = le.inverse_transform([pred_idx])[0]
    if max_prob < CONFIDENCE_THRESHOLD:
        return "Unknown", round(max_prob * 100, 1)
    return name, round(max_prob * 100, 1)

def recognize_image(image_path: str, mtcnn, resnet, head, clf, le):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"[ERROR] Cannot read image: {image_path}")
        return

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    boxes, _ = mtcnn.detect(pil_img)

    if boxes is None:
        cv2.imshow("Result", img_bgr)
        cv2.waitKey(0)
        return

    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(img_bgr.shape[1], int(box[2])), min(img_bgr.shape[0], int(box[3]))
        
        face_rgb = img_rgb[y1:y2, x1:x2]
        if face_rgb.size == 0: continue
        
        face_pil = Image.fromarray(face_rgb).resize(IMAGE_SIZE)
        emb = get_embedding(mtcnn, resnet, head, np.array(face_pil))
        
        label, confidence = predict_face(emb, clf, le) if emb is not None else ("Unknown", 0.0)
        
        color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_bgr, f"{label} {confidence}%", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imshow("Face Recognition Result", img_bgr)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def recognize_webcam(mtcnn, resnet, head, clf, le):
    cap = cv2.VideoCapture(0)
    print("[LIVE] Press 'q' to quit.")
    frame_skip = 2
    frame_count = 0
    last_results = []

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        if frame_count % frame_skip == 0:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            boxes, _ = mtcnn.detect(Image.fromarray(img_rgb))
            last_results = []
            if boxes is not None:
                for box in boxes:
                    x1, y1, x2, y2 = [int(v) for v in box]
                    face_rgb = img_rgb[max(0,y1):min(frame.shape[0],y2), max(0,x1):min(frame.shape[1],x2)]
                    if face_rgb.size == 0: continue
                    face_pil = Image.fromarray(face_rgb).resize(IMAGE_SIZE)
                    emb = get_embedding(mtcnn, resnet, head, np.array(face_pil))
                    label, confidence = predict_face(emb, clf, le) if emb is not None else ("Unknown", 0.0)
                    last_results.append((x1, y1, x2, y2, label, confidence))

        for (x1, y1, x2, y2, label, conf) in last_results:
            color = (0, 255, 0) if label != "Unknown" else (0, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"{label} {conf}%", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow("Webcam", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()

# ─────────────────────────────────────────────
#  MAIN ENTRY POINT (Jupyter-Compatible)
# ─────────────────────────────────────────────
def main():
    import argparse
    import sys

    parser = argparse.ArgumentParser(
        description="CNN Facial Recognition System (v2 – DeconvSkip Head)"
    )
    parser.add_argument(
        "--mode",
        choices=["train", "image", "webcam", "train+webcam", "train+image"],
        default="train+webcam",
        help=(
            "train         → Train model from faces/ folder\n"
            "image         → Recognize face in a static image\n"
            "webcam        → Live webcam recognition\n"
            "train+webcam  → Train then launch webcam (default)\n"
            "train+image   → Train then test on an image"
        )
    )
    parser.add_argument("--image", type=str, help="Path to image for 'image' mode")
    
    # Use parse_known_args to handle Jupyter's internal '--f' arguments
    args, unknown = parser.parse_known_args()

    # Load CNN backbone + deconv head
    print("[INIT] Loading FaceNet + MTCNN + DeconvSkipHead...")
    mtcnn, resnet, head = load_models()
    print("[INIT] Models loaded.\n")

    if args.mode in ("train", "train+webcam", "train+image"):
        clf, le = train(FACES_DIR, mtcnn, resnet, head)
    else:
        clf, le = load_classifier()
        print(f"[INFO] Loaded classifier from {MODEL_SAVE_PATH}. Known persons: {list(le.classes_)}")

    if args.mode in ("image", "train+image"):
        img_path = args.image
        if not img_path:
            img_path = input("\nEnter path to test image: ").strip()
        recognize_image(img_path, mtcnn, resnet, head, clf, le)

    elif args.mode in ("webcam", "train+webcam"):
        recognize_webcam(mtcnn, resnet, head, clf, le)


if __name__ == "__main__":
    main()

[INFO] Using device: cpu
[INIT] Loading FaceNet + MTCNN + DeconvSkipHead...
[INFO] DeconvSkipHead loaded.
[INIT] Models loaded.


[TRAIN] Found 10 person(s): ['aditya joshi', 'aditya kumar', 'amit', 'bhawna', 'harsh', 'harshit', 'pranjal', 'ravi', 'rupesh_kattebaaz', 'yadav']
  → aditya joshi: 32 image(s)
  → aditya kumar: 35 image(s)
  → amit: 3 image(s)
  → bhawna: 25 image(s)
  → harsh: 5 image(s)
  → harshit: 7 image(s)
  → pranjal: 6 image(s)
  → ravi: 42 image(s)
  → rupesh_kattebaaz: 9 image(s)
  → yadav: 9 image(s)

[TRAIN] Classification Report:
                  precision    recall  f1-score   support

    aditya joshi       1.00      1.00      1.00         6
    aditya kumar       1.00      1.00      1.00         7
            amit       0.00      0.00      0.00         1
          bhawna       1.00      1.00      1.00         5
           harsh       0.50      1.00      0.67         1
         harshit       1.00      1.00      1.00         1
         pranjal       1.00     

c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

[LIVE] Press 'q' to quit.


In [2]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
!pip install facenet-pytorch

  Using cached facenet_pytorch-2.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Aditya Joshi\AppData\Local\Temp\pip-install-vbof8idg\numpy_31212ea943354179bb8bd8f18cd1ae7f\vendored-meson\meson\meson.py setup C:\Users\Aditya Joshi\AppData\Local\Temp\pip-install-vbof8idg\numpy_31212ea943354179bb8bd8f18cd1ae7f C:\Users\Aditya Joshi\AppData\Local\Temp\pip-install-vbof8idg\numpy_31212ea943354179bb8bd8f18cd1ae7f\.mesonpy-cxznxfjt -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\Aditya Joshi\AppData\Local\Temp\pip-install-vbof8idg\numpy_31212ea943354179bb8bd8f18cd1ae7f\.mesonpy-cxznxfjt\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\Aditya Joshi\AppData\Local\Temp\pip-install-vbof8idg\numpy_31212ea943354179bb8bd8f18cd1ae7f
      Build dir: C:\User

In [3]:
pip install torch torchvision facenet-pytorch opencv-python numpy==2.2.0

  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Aditya Joshi\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\~umpy.libs\\libopenblas64__v0.3.23-293-gc2f4bdbb-gcc_10_3_0-2bde3a66a51006b2b53eb373ff767a3f.dll'
Consider using the `--user` option or check the permissions.

You should consider upgrading via the 'c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
pip install --user numpy==2.2.0 opencv-python facenet-pytorch

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
  Using cached facenet_pytorch-2.6.0-py3-none-any.whl (1.9 MB)
  Using cached pillow-10.2.0-cp310-cp310-win_amd64.whl (2.6 MB)
  Using cached facenet_pytorch-2.5.3-py3-none-any.whl (1.9 MB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Aditya Joshi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.
